# 2 · Training LeanNPE (small version)

LeanNPE = strain encoder → 256-D context (+ a 32-D **rank embedding**) → neural
spline flow over 11 parameters. One forward pass gives the posterior of the
k-th loudest signal. This notebook trains the *production architecture* for a
few epochs on the notebook-1 dataset — enough to watch the training signals that
matter, not to get a useful model. The real run is `experiments/train_lean_npe.py`
(40 epochs, 200k events, ~3 h on an H100).

Design rules (each reverses a measured failure — see the README):
- **no normalisation of the strain** (whitened amplitude *is* the distance cue);
- **rank conditioning** for overlapping signals;
- **pure negative-log-likelihood** objective.

In [ ]:
import os, sys, json, subprocess
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch

# run from the repo root so relative paths (configs/, data/, model/) resolve
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
for p in (ROOT, ROOT / "src", ROOT / "experiments"):  # experiments/: remix_data imports delay_aug
    sys.path.insert(0, str(p))
import logging; logging.getLogger("bilby").setLevel(logging.WARNING)

DATA = ROOT / "data" / "notebook_demo"      # small demo dataset (git-ignored)
MODEL_DIR = ROOT / "model" / "notebook_demo" # small demo model (git-ignored)
DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("repo:", ROOT, "| device:", DEVICE)

## The model

Production recipe: coherent (cross-detector) encoder, matched-filter mass tokens,
PSD conditioning, flow coordinates (chirp mass, mass ratio). `dist_range` sets
the distance support; labels outside it are clamped, so it must cover the data.

In [ ]:
from ahsd.models.lean_npe import LeanNPE, PARAM_NAMES, ParamScaler
from experiments.remix_data import RemixDataset

ARGS = dict(premerger=False, psd_cond=True, psd_bands=16, encoder_type="coherent",
            mass_param="mc_q", mf_tokens=True, geom_dropout=0.25)
torch.manual_seed(0)
model = LeanNPE(**ARGS, dist_range=(8.0, 6000.0)).to(DEVICE)
print(f"LeanNPE: {sum(p.numel() for p in model.parameters()):,} parameters")
print("distance support:", model.scaler.dist_range, "Mpc")

# the scaler is a FIXED invertible map physical <-> [-1, 1] (log for masses/distance)
x = torch.tensor([[36.0, 29.0, 440.0, 1.0, -1.2, 2.7, 1.0, 1.0, 0.0, 0.3, 0.2]])
y = model.scaler.normalize(x.to(DEVICE))
np.set_printoptions(suppress=True, precision=3)
print("normalized:", y.cpu().numpy()[0], "\nround trip:", model.scaler.denormalize(y).cpu().numpy()[0])

## Data loaders

Training data is **remixed** (fresh noise/shift/distance each epoch); validation is
fixed so epochs are comparable. `persistent_workers` must stay off so `set_epoch`
reaches the workers.

In [ ]:
bank = ROOT / "data" / "noise_bank"
train_ds = RemixDataset(str(DATA / "memmap" / "train"), remix=True,
                        real_noise_dir=str(bank) if bank.exists() else None,
                        real_noise_prob=0.9 if bank.exists() else 0.0,
                        det_dropout=0.2, delay_aug=0.5, return_asd_bands=True)
val_ds = RemixDataset(str(DATA / "memmap" / "validation"), remix=False, seed=1234, return_asd_bands=True)
train_dl = torch.utils.data.DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=0)
val_items = [val_ds[i] for i in range(len(val_ds))]
val_batch = [torch.stack([it[k] for it in val_items]) for k in range(5)]
print(len(train_ds), "train events,", len(val_ds), "validation events")

## The loss: one encoder pass, the flow queried once per rank

For an event with *n* signals the encoder runs once; the flow scores the true
parameters of rank 0, 1, …, n−1 under the same context (as `batch_nll` in the trainer).

In [ ]:
def batch_nll(model, strain, params, nsig, asd, context=None):
    context = model.encode(strain, asd) if context is None else context
    losses, count = [], 0
    for r in range(int(nsig.max())):
        idx = (nsig > r).nonzero(as_tuple=True)[0]
        if idx.numel() == 0:
            continue
        rank = torch.full((idx.numel(),), r, dtype=torch.long, device=strain.device)
        losses.append(model.nll(None, params[idx, r], rank, context=context[idx]).sum())
        count += idx.numel()
    return torch.stack(losses).sum() / count


@torch.no_grad()
def diagnostics(model, batch, n_post=128):
    """val NLL, shuffle test, and spurious railing — the per-epoch health checks."""
    model.eval()
    strain, pv, nsig, snr, asd = [b.to(DEVICE) for b in batch]
    ctx = model.encode(strain.float(), asd.float())
    nll = batch_nll(model, strain, pv, nsig, asd, context=ctx).item()
    # shuffle test: give each event ANOTHER event's context. If NLL barely rises,
    # the encoder is being ignored (a marginal fit) — the failure that retired
    # the previous model. It must climb several nats.
    shuf = batch_nll(model, strain, pv, nsig, asd, context=ctx[torch.randperm(len(ctx))]).item()
    # spurious railing: posterior samples pinned at a bound the truth is not near
    rank0 = torch.zeros(len(ctx), dtype=torch.long, device=DEVICE)
    fc = model._full_context(ctx, rank0)
    y, _ = model.flow.inverse(torch.randn(len(ctx) * n_post, len(PARAM_NAMES), device=DEVICE),
                              fc.repeat_interleave(n_post, 0))
    y = model.scaler.wrap(y).view(len(ctx), n_post, -1)
    yt = model.scaler.normalize(pv[:, 0]).unsqueeze(1)
    rail = (((y > 0.999) & (yt < 0.9)) | ((y < -0.999) & (yt > -0.9))) & ~model.scaler.circ_mask
    model.train()
    return nll, shuf - nll, rail.any(-1).float().mean().item()

## Train

AdamW with a cosine schedule **sized to the real stopping epoch** (a schedule sized for
140 epochs and stopped at 19 left the learning rate near its peak and over-sharpened the
flow — the "validation improves while real events get worse" bug). Checkpoints are kept
only when spurious railing is below 10% (calibration-gated selection, not raw NLL).

In [ ]:
EPOCHS, LR = 10, 3e-4
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
steps = EPOCHS * len(train_dl)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=steps, pct_start=0.1, anneal_strategy="cos")
MODEL_DIR.mkdir(parents=True, exist_ok=True)
history, best, best_any = [], None, None


def save(epoch, val, rule):
    torch.save({"model_state_dict": model.state_dict(), "epoch": epoch, "val_nll": val,
                "args": {**ARGS, "dataset_distance_prior": "uniform",
                         "dataset_event_types": ["BBH", "BNS", "NSBH"]},
                "diagnostics": {**history[-1], "selected_by": rule}}, MODEL_DIR / "best_model.pth")
for epoch in range(1, EPOCHS + 1):
    train_ds.set_epoch(epoch)
    model.train(); tl = []
    for strain, pv, nsig, snr, asd in train_dl:
        strain, pv, nsig, asd = strain.to(DEVICE).float(), pv.to(DEVICE), nsig.to(DEVICE), asd.to(DEVICE).float()
        loss = batch_nll(model, strain, pv, nsig, asd)
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 10.0)
        opt.step(); sched.step(); tl.append(loss.item())
    val, shuf, rail = diagnostics(model, val_batch)
    history.append(dict(epoch=epoch, train=np.mean(tl), val=val, shuf=shuf, rail=rail))
    gate = rail <= 0.10
    note = ""
    if gate and (best is None or val < best):
        best = val; save(epoch, val, "calibration gate"); note = "  -> saved (passed railing gate)"
    elif best is None and (best_any is None or val < best_any):
        # like the trainer: until some epoch passes the gate, keep the best-NLL
        # epoch as a fallback so there is always a usable checkpoint
        best_any = val; save(epoch, val, "best-NLL fallback (no epoch passed the gate)")
        note = "  -> saved as fallback (railing above gate)"
    print(f"epoch {epoch} | train {np.mean(tl):6.3f} | val {val:6.3f} | shufΔ {shuf:+.2f} | rail {rail:.2f}{note}")

In [ ]:
h = {k: [e[k] for e in history] for k in history[0]}
fig, ax = plt.subplots(1, 3, figsize=(14, 3.2))
ax[0].plot(h["epoch"], h["train"], label="train"); ax[0].plot(h["epoch"], h["val"], label="val"); ax[0].set(title="NLL"); ax[0].legend()
ax[1].plot(h["epoch"], h["shuf"]); ax[1].set(title="shuffle ΔNLL (must climb)")
ax[2].plot(h["epoch"], h["rail"]); ax[2].axhline(0.10, ls="--", c="k"); ax[2].set(title="spurious railing (gate 0.10)")
for a in ax: a.set_xlabel("epoch")
plt.tight_layout(); plt.show()

**Reading it:** NLL falls; the **shuffle ΔNLL** climbing means the encoder's context
actually carries event information (pinned near 0 = the network ignores the data);
railing should fall under the gate. At this scale (~800 events, 10 epochs) it does
not happen yet: NLL falls but the shuffle ΔNLL stays near 0 — the model is still
fitting the *population* (a marginal fit) rather than reading each event, and
railing stays above the gate, so the saved checkpoint is the best-NLL fallback.
That is the expected behaviour of a tiny run, and exactly what these two
diagnostics are for. The production 200k-event run passes the gate at epoch 1 and
reaches shuffle ΔNLL ≈ 20.

**Full scale:** `sbatch --export=ALL,NAME=lean_npe_v3,DATA=data/dataset_v3,EPOCHS=40 scripts/slurm/train_lean_npe.sbatch`
(the job then certifies the checkpoint with `scripts/validate_checkpoint.py`).
Next: `03_results_analysis.ipynb`.